# Lecture 07 - Solutions to Part B (Python Applications)

This notebook gives worked Python solutions for Exercises 17-21. The focus is precision: standard errors, confidence intervals for means, interval width, and careful interpretation.

Throughout the notebook, confidence intervals use normal critical values as a working approximation, matching the lecture.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:.6f}".format)
plt.style.use("seaborn-v0_8-whitegrid")

data_path = "../data/"  # Use your data here. Keep the final slash.

aapl = pd.read_csv(data_path + "Lecture 07 aapl daily returns 2012 2021.csv")
nflx = pd.read_csv(data_path + "Lecture 07 nflx daily log returns 2015 2026.csv")
credit = pd.read_csv(data_path + "Lecture 07 german credit.csv")

aapl["Date"] = pd.to_datetime(aapl["Date"])
nflx["date"] = pd.to_datetime(nflx["date"])

Z_VALUES = {0.90: 1.645, 0.95: 1.96, 0.99: 2.58}

def mean_interval(series, level=0.95, label="value"):
    x = pd.to_numeric(series, errors="coerce").dropna()
    n = x.size
    mean = x.mean()
    sd = x.std(ddof=1)
    se = sd / np.sqrt(n)
    z = Z_VALUES[level]
    margin = z * se
    return pd.Series({
        "label": label,
        "level": level,
        "n": n,
        "mean": mean,
        "sd": sd,
        "se": se,
        "z": z,
        "margin": margin,
        "lower": mean - margin,
        "upper": mean + margin,
    })

---

## Exercise 17 - Confidence Interval for Apple Mean Daily Return

In [ ]:
apple_95 = mean_interval(aapl["daily_return"], level=0.95, label="AAPL daily return")
apple_95

In [ ]:
annualized = pd.Series({
    "annualized_point_estimate": apple_95["mean"] * 252,
    "annualized_lower": apple_95["lower"] * 252,
    "annualized_upper": apple_95["upper"] * 252,
})

annualized

**Interpretation.** The interval describes uncertainty around the mean daily return, not the range of individual daily returns. Annualizing by multiplying by 252 is only a linear reporting convention for the mean and its interval bounds. It ignores compounding and should not be confused with annualizing volatility.

---

## Exercise 18 - Confidence Level Sensitivity

In [ ]:
level_rows = []
for level in [0.90, 0.95, 0.99]:
    row = mean_interval(aapl["daily_return"], level=level, label="AAPL daily return")
    level_rows.append(row)

level_table = pd.DataFrame(level_rows)
level_table["width"] = level_table["upper"] - level_table["lower"]
level_table[["level", "mean", "se", "z", "lower", "upper", "width"]]

In [ ]:
plt.figure(figsize=(8, 3.8))
y = np.arange(len(level_table))
plt.hlines(y, level_table["lower"], level_table["upper"], lw=4)
plt.scatter(level_table["mean"], y, color="black", zorder=3)
plt.yticks(y, [f"{int(level * 100)}%" for level in level_table["level"]])
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.title("AAPL Mean Daily Return: Confidence Level Sensitivity")
plt.xlabel("Daily return")
plt.ylabel("Confidence level")
plt.show()

**Interpretation.** The point estimate and standard error do not change across rows. The interval widens because the critical value is larger for higher confidence levels. This is the precision-confidence tradeoff.

---

## Exercise 19 - Compare Apple and Netflix Precision

In [ ]:
asset_intervals = pd.DataFrame([
    mean_interval(aapl["daily_return"], level=0.95, label="AAPL simple daily return"),
    mean_interval(nflx["log_return"], level=0.95, label="NFLX daily log return"),
])
asset_intervals["width"] = asset_intervals["upper"] - asset_intervals["lower"]
asset_intervals[["label", "n", "mean", "sd", "se", "lower", "upper", "width"]]

In [ ]:
plt.figure(figsize=(8, 3.8))
y = np.arange(len(asset_intervals))
plt.hlines(y, asset_intervals["lower"], asset_intervals["upper"], lw=4)
plt.scatter(asset_intervals["mean"], y, color="black", zorder=3)
plt.yticks(y, asset_intervals["label"])
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.title("Approximate 95% Intervals for Mean Daily Returns")
plt.xlabel("Daily return")
plt.show()

**Interpretation.** The comparison is about estimation precision, not about which asset earns more. The standard error depends on both the volatility of individual returns and the sample size. A more volatile return series can have a wider interval even with many observations.

---

## Exercise 20 - Confidence Interval for Mean Loan Amount

In [ ]:
amount_95 = mean_interval(credit["amount"], level=0.95, label="Mean loan amount")
amount_95

In [ ]:
amount_summary = credit["amount"].describe()
amount_summary

**Interpretation.** The confidence interval describes uncertainty around the mean loan amount for the population or process represented by the sample. It does not describe the range of individual loan amounts. Reporting the sample average alone gives a point estimate; the confidence interval adds information about precision.

---

## Exercise 21 - Careful Precision Memo

In [ ]:
memo = f"""
For Apple daily returns, the estimated mean daily return is {apple_95['mean']:.4%}.
The estimated standard error is {apple_95['se']:.4%}, giving an approximate 95% confidence interval from {apple_95['lower']:.4%} to {apple_95['upper']:.4%}.
The standard deviation describes variation across individual daily returns; the standard error describes uncertainty in the estimated mean return.
This interval is based on historical sample data and the normal-approximation method used in the lecture, so it should be interpreted as a plausible range for the unknown mean daily return, not as a prediction range for future daily returns.
"""
print(memo)